# Stage 6 - Export app-consumable artifacts

**NASA Space Apps 2026 - MODIS/VIIRS hotspot harmonization**

Integrity-verified artifact loads, the end-to-end inference export (harmonized daily series, calendar, anomalies, optional 14-day forecast), contract validation, and the final handoff manifest the Streamlit app consumes on its separate host.

| | |
|---|---|
| Stage | 6 of 6 - offline compute |
| Network required | NO |
| Runs | the committed CLI scripts (`scripts/`) - logic stays out of cells |
| Writes to | `out/inference/ + export manifest` |

Rules that make this notebook reproducible (docs/DEVELOPMENT.md): record the
environment below, keep every output under the working directory, never fetch
from PyPI/GitHub/Hugging Face/REST APIs during offline compute, and keep
generated weights/datasets out of Git.


## 0. Environment

In [ ]:
import sys, platform, subprocess
from pathlib import Path

# Kaggle working dir is the ONLY writable location during offline compute.
KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "out"
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd()

print("python   :", sys.version)
print("platform :", platform.platform())
print("machine  :", platform.machine())
print("repo     :", REPO_ROOT)
print("work dir :", WORK_DIR)

RECORD = {"python": sys.version.split()[0], "platform": platform.platform(), "machine": platform.machine()}


## 0.1 Resolve paths

Attach the stage-4/stage-5 outputs and the FIRMS archive; helpers wrap the committed CLIs so logic stays out of cells.

In [ ]:
import json, hashlib
from pathlib import Path

def run_cli(script: str, *args: str, expect: int = 0) -> subprocess.CompletedProcess:
    """Run a committed CLI script; fail the notebook loudly on bad exits."""
    cmd = [sys.executable, str(REPO_ROOT / "scripts" / script), *args]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=7200, cwd=str(REPO_ROOT))
    if proc.returncode != expect:
        print(proc.stdout[-4000:]); print(proc.stderr[-4000:])
        raise RuntimeError(f"{script} exited {proc.returncode}, expected {expect}")
    print(proc.stdout[-2000:])
    return proc

def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def latest_artifact(base: Path, prefix: str) -> Path:
    dirs = sorted((p for p in base.glob(prefix + "_*") if p.is_dir()), key=lambda p: p.stat().st_mtime)
    if not dirs:
        raise FileNotFoundError(f"no {prefix}_* artifact under {base}; did the training step run?")
    return dirs[-1]


SOURCE_DIR = Path("/kaggle/input") / "firms"
if not SOURCE_DIR.exists():
    SOURCE_DIR = REPO_ROOT / "tests" / "fixtures" / "sample_aoi"  # synthetic demo
AOI = "bbox:-123.5,39.5,-120.5,42.0"
harmonizer_dir = latest_artifact(WORK_DIR / "harmonization", "harmonizer")
forecaster_dir = latest_artifact(WORK_DIR / "calendar", "forecaster")
print("harmonizer:", harmonizer_dir)
print("forecaster:", forecaster_dir)


## 1. One-command end-to-end alternative

`scripts/run_pipeline.py` chains validation -> training -> forecasting -> inference with stage-by-stage timing recorded in `pipeline_run.json`; and `scripts/run_demo.py` additionally renders a demo report. Use those for the full journey; the cells below run the export stage directly.

In [ ]:

inference_out = WORK_DIR / "inference"
proc = run_cli("run_inference.py", "--source-dir", str(SOURCE_DIR),
               "--harmonizer-dir", str(harmonizer_dir), "--forecaster-dir", str(forecaster_dir),
               "--output-dir", str(inference_out), "--aoi", AOI, "--seed", "2026")


## 2. Verify contracts and integrity

Loading an artifact verifies its recorded SHA-256; the exported calendar/forecast frame is validated against the column contract (`value_type`, `quality_state`, quantile ordering).

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(REPO_ROOT))
from src.artifacts.serialization import load_harmonizer_bundle, load_forecaster_bundle, load_manifest
from src.calendar.records import validate_calendar_forecast_frame

bundle = load_harmonizer_bundle(harmonizer_dir)      # raises ArtifactIntegrityError on hash mismatch
state = load_forecaster_bundle(forecaster_dir)
manifest = load_manifest(harmonizer_dir)
print("harmonizer:", manifest["model"]["name"], manifest["model"]["version"])

records = pd.read_csv(inference_out / "calendar_forecast_records.csv")
validate_calendar_forecast_frame(records)            # raises on contract violations
print("calendar/forecast contract: VALID")
print(records["quality_state"].value_counts().to_dict())
print(records[records["quality_state"] == "forecast"].head())


## 3. Handoff manifest and app startup

The exported records + provenance manifests move to the app host as versioned outputs. The separate Streamlit app reads only these files (no network, no recomputation) and must label observed / harmonized / forecast / cached data distinctly.

In [ ]:
import json, hashlib
from pathlib import Path

export_manifest = {
    "schema_version": "1.0.0",
    "app_inputs": {
        str(p.relative_to(inference_out)): {"sha256": sha256(p), "bytes": p.stat().st_size}
        for p in sorted(inference_out.rglob("*")) if p.is_file()
    },
    "app_contract": {
        "calendar_records": "calendar_forecast_records.csv",
        "harmonized_daily": "harmonized_daily.csv",
        "anomalies": "anomaly_records.csv",
        "labels": ["observed_viirs", "harmonized_estimate", "forecast", "missing_coverage"],
        "rules": [
            "The app renders only; it never retrains or fetches data.",
            "Missing coverage must display as missing, never as zero activity.",
            "Forecast intervals are experimental/uncalibrated until validated.",
        ],
    },
}
(WORK_DIR / "export_manifest.json").write_text(json.dumps(export_manifest, indent=2, sort_keys=True))
print(f"{len(export_manifest['app_inputs'])} exported files recorded")
print("handoff manifest:", WORK_DIR / "export_manifest.json")
